In [1]:
import os
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

DATA_DIR = "/kaggle/input/datasets/veeraiahkondra/vnkt11/Final_Data_CLAHE"

train_transform = transforms.Compose([
    transforms.Resize((299,299)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((299,299)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR,"train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR,"val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR,"test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)
test_loader  = DataLoader(test_dataset, batch_size=8)

class_names = train_dataset.classes

In [2]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

inception = models.inception_v3(weights="IMAGENET1K_V1")  # ⚠ do NOT set aux_logits=False

# Replace classifiers
inception.fc = nn.Linear(inception.fc.in_features, 4)
inception.AuxLogits.fc = nn.Linear(inception.AuxLogits.fc.in_features, 4)

inception = inception.to(device)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 178MB/s] 


In [3]:
import torch.optim as optim

class_weights = torch.tensor([1.0,1.0,1.2,1.2]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = optim.AdamW(inception.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()
EPOCHS = 30

for epoch in range(EPOCHS):

    inception.train()
    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = inception(images)

            if isinstance(outputs, tuple):
                main, aux = outputs
                loss = criterion(main, labels) + 0.4 * criterion(aux, labels)
            else:
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

/tmp/ipykernel_57/3167120380.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_57/3167120380.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
import numpy as np

def evaluate_model(model, loader):

    model.eval()

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 🔥 Run evaluation again (IMPORTANT)
y_true, y_pred, y_prob = evaluate_model(effnet, test_loader)

# Assign to expected variables
all_labels = y_true
all_preds = y_pred

# ==============================
# 📊 CONFUSION MATRIX
# ==============================
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import numpy as np

def evaluate_model(model, loader):

    model.eval()

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
from sklearn.metrics import *
import numpy as np

def compute_metrics(y_true, y_pred, y_prob):

    cm = confusion_matrix(y_true, y_pred)

    specificity = []
    for i in range(len(cm)):
        tn = np.sum(cm) - (np.sum(cm[i,:]) + np.sum(cm[:,i]) - cm[i,i])
        fp = np.sum(cm[:,i]) - cm[i,i]
        specificity.append(tn / (tn + fp + 1e-6))

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='macro'),
        "Recall": recall_score(y_true, y_pred, average='macro'),
        "F1": f1_score(y_true, y_pred, average='macro'),
        "Balanced_Accuracy": recall_score(y_true, y_pred, average='macro'),
        "Specificity": np.mean(specificity),
        "AUC": roc_auc_score(y_true, y_prob, multi_class='ovr'),
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

In [ ]:
y_true, y_pred, y_prob = evaluate_model(inception, test_loader)

metrics = compute_metrics(y_true, y_pred, y_prob)

for k,v in metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_true, classes=[0,1,2,3])

for i in range(4):
    fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
    plt.plot(fpr, tpr, label=class_names[i])

plt.legend()
plt.title("ROC Curve (InceptionV3)")
plt.show()

In [ ]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report

# 🔥 USE CORRECT MODEL
model = effnet   # 👈 FIX HERE

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        if isinstance(outputs, tuple):
            outputs = outputs[0]

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ==============================
# 📊 METRICS
# ==============================
f1 = f1_score(all_labels, all_preds, average='macro')

print(f"🔥 TEST F1: {f1:.4f}")

print("\n📊 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2

def gradcam(model, image):

    model.eval()

    features = []
    gradients = []

    # 🔥 Hooks
    def forward_hook(module, input, output):
        features.append(output)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    # EfficientNet last conv layer
    target_layer = model.features[-1]

    handle_fwd = target_layer.register_forward_hook(forward_hook)
    handle_bwd = target_layer.register_full_backward_hook(backward_hook)

    image = image.unsqueeze(0).to(device)

    output = model(image)

    if isinstance(output, tuple):
        output = output[0]

    pred = output.argmax(dim=1)

    model.zero_grad()
    output[0, pred].backward()

    # 🔥 Get stored data
    grad = gradients[0]
    fmap = features[0]

    weights = grad.mean(dim=(2,3), keepdim=True)

    cam = (weights * fmap).sum(dim=1).squeeze()

    cam = cam.detach().cpu().numpy()

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (224,224))

    # 🔥 Normalize safely
    if cam.max() != 0:
        cam = cam / cam.max()

    # 🔥 Remove hooks (IMPORTANT)
    handle_fwd.remove()
    handle_bwd.remove()

    return cam

In [ ]:
def rise(model, image, N=1000, p=0.5):

    model.eval()

    image = image.to(device)

    # 🔥 Get predicted class
    with torch.no_grad():
        output = model(image.unsqueeze(0))
        if isinstance(output, tuple):
            output = output[0]
        pred = output.argmax(dim=1).item()

    # 🔥 Generate masks
    masks = torch.bernoulli(torch.full((N,1,224,224), p)).to(device)

    preds = []

    with torch.no_grad():
        for i in range(N):

            masked = image * masks[i]

            out = model(masked.unsqueeze(0))

            if isinstance(out, tuple):
                out = out[0]

            prob = torch.softmax(out, dim=1)[0, pred]

            preds.append(prob.item())

    preds = torch.tensor(preds).to(device)

    # 🔥 Compute heatmap
    heatmap = (masks.squeeze() * preds.view(-1,1,1)).mean(dim=0)

    heatmap = heatmap.cpu().numpy()

    # 🔥 Normalize
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap = heatmap / heatmap.max()

    return heatmap

In [ ]:
img, label = test_dataset[0]

cam = gradcam(effnet, img)
rise_map = rise(effnet, img)

plt.imshow(cam, cmap='jet')
plt.title("Grad-CAM")
plt.show()

plt.imshow(rise_map, cmap='jet')
plt.title("RISE")
plt.show()